# Outbound-path duration — FbR, Test, Day 44 (two-session comparison)

Per-trial **duration of the outbound path** for **FbR_M01569522**, Test tree, **Day 44**
(`Day44-52FloorRotation`), comparing `2026-07-13T151307Z` (S1) and `2026-07-13T153226Z` (S2).

**Two outbound-duration definitions are compared, in clearly separated sections below:**
- **Definition A — original** — outbound = trial start → target-zone **triggered**
  (`outbound_dur` = `TTT` = `tz_triggered_time − start_time`, the time to actually *reach* the zone).
  Sections **A1–A3**.
- **Definition B — NEW** — outbound = trial start → target-zone **available**
  (`outbound_dur_avail` = `tz_available_time − start_time`): start until the zone is *announced*
  (the pre-availability phase). This is a **different leg** — it stops before the approach
  `available → triggered`, so A and B are not directly comparable magnitudes — and, unlike A, it is
  defined even on a true Miss (the zone is announced whether or not the animal reaches it).
  Mirrored in sections **B1–B3** at the end.

**Definition A detail.** A trial that never triggers the zone (a true Miss, `tz_triggered_time` = NaN)
has no inbound leg, so its whole span (`end_time − start_time`) counts as outbound instead; here every
trial triggered the zone, so outbound duration = `TTT` throughout. The complementary leg is the
**poke latency** (`TTP` = `end_time − tz_triggered_time`, target → poke), reported for context.

Views (mirroring the reference notebooks), applied to **each** definition: **(1)** distribution —
histogram, box + strip, ECDF, with summary stats + Mann–Whitney U; **(2)** within-session trend (does
it climb across trials?); **(3)** success rate as a function of the duration.

> *Outbound **distance** (path **length** = summed DLC-centroid displacement via
> `movement.kinematics.compute_path_length`, as in `Requested_plots.ipynb` Fig 2) is omitted: these
> two sessions have raw video only — no DLC pose has been generated — so path length can't be
> computed yet. Duration-only for now.*

In [ ]:
# Setup.
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats

from data_conduit.refactor_qc import qc_datastructure

# These two sessions live on /media/sepi/Elements  (NOT Elements1 - that drive only syncs to ~2026-07-06).
# Layout: Test/<mouseID>/<day>/<session>, so depth=2 with level_names=('mouseID', 'day').
ROOT = pathlib.Path('/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Test')
MOUSE = 'FbR_M01569522'
SESSIONS = ['2026-07-15T171049Z', '2026-07-15T171740Z']          # Day 44 = Day44-52FloorRotation
LABELS   = {SESSIONS[0]: 'S1  17:10:49', SESSIONS[1]: 'S2  17:17:40'}
COLOURS  = {SESSIONS[0]: '#2e7ebc', SESSIONS[1]: '#e53592'}

In [ ]:
# Load the events stream only (lightest: just the distilled trial table) for the two sessions and
# compute BOTH per-trial outbound-duration definitions. `include` matches on the session-timestamp folder.
trials = qc_datastructure(root=ROOT, depth=2, level_names=('mouseID', 'day'),
                          streams=('events',), include=SESSIONS).load()['trials']

t = trials[trials['session'].isin(SESSIONS)].copy()
t['reached_target']     = t['outbound_end_time'].notna()                                   # target zone triggered?
# Definition A (ORIGINAL): outbound = start -> target-zone TRIGGERED (= TTT; whole trial on a true Miss).
t['outbound_dur']       = t['outbound_end_time'].fillna(t['end_time']) - t['start_time']
# Definition B (NEW):      outbound = start -> target-zone AVAILABLE (zone announced; defined even on a Miss).
t['outbound_dur_avail'] = t['tz_available_time'] - t['start_time']
t['poke_latency']       = t['TTP']                                                         # target -> poke (context leg)
t['success']            = (t['outcome'] == 'Success').astype(int)
t = t.sort_values(['session', 'trial_index']).reset_index(drop=True)

sub = {s: t[t['session'] == s] for s in SESSIONS}                                          # per-session views used below
for s in SESSIONS:
    print(f"{LABELS[s]}   trials={len(sub[s]):3d}   outcomes={sub[s]['outcome'].value_counts().to_dict()}")
print(f"\ntrials that never triggered the target zone (true Miss): {int((~t['reached_target']).sum())}")

display(t[['session', 'trial_index', 'outcome', 'reached_target', 'start_time',
           'tz_available_time', 'outbound_end_time', 'end_time',
           'outbound_dur', 'outbound_dur_avail', 'poke_latency']])

---
## Definition A (original) — outbound = start → target-zone **triggered** (`TTT`)

Sections **A1–A3** use **`outbound_dur`** (`tz_triggered_time − start_time`) — the original
"time to target" metric. The parallel **Definition B (NEW)** section further down repeats all three
on `start → available`.

In [ ]:
#=== 1. Outbound-duration distribution - summary stats + Mann-Whitney
# Durations are right-skewed, so compare with a rank test (Mann-Whitney U) rather than a t-test.
def describe(v, s):
    v = v.dropna()
    return {'session': LABELS[s], 'n': len(v), 'mean': v.mean(), 'median': v.median(),
            'std': v.std(), 'q25': v.quantile(.25), 'q75': v.quantile(.75), 'min': v.min(), 'max': v.max()}

a, b = sub[SESSIONS[0]]['outbound_dur'].dropna(), sub[SESSIONS[1]]['outbound_dur'].dropna()
summary = pd.DataFrame([describe(sub[s]['outbound_dur'], s) for s in SESSIONS]).set_index('session')
u, p_mwu = stats.mannwhitneyu(a, b, alternative='two-sided')

pd.set_option('display.float_format', lambda x: f'{x:.2f}')
print('OUTBOUND-PATH DURATION (start -> target-zone triggered = TTT), seconds\n')
print(summary.to_string())
print(f'\nMann-Whitney U: U={u:.0f}, p={p_mwu:.3g}  |  '
      f'median {a.median():.2f}s vs {b.median():.2f}s  ({b.median() / a.median():.2f}x)')
pa, pb = sub[SESSIONS[0]]['poke_latency'].dropna(), sub[SESSIONS[1]]['poke_latency'].dropna()
_, p_pl = stats.mannwhitneyu(pa, pb, alternative='two-sided')
print(f'context - poke latency (TTP, target -> poke) median {pa.median():.2f} -> {pb.median():.2f}s  (p={p_pl:.3g})')

# --- Plot 1: histogram, box + strip, ECDF
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
bins = np.linspace(0, np.nanpercentile(pd.concat([a, b]), 99) * 1.05, 25)
for s, v in [(SESSIONS[0], a), (SESSIONS[1], b)]:
    ax[0].hist(v, bins=bins, alpha=.55, color=COLOURS[s], edgecolor='white', label=f'{LABELS[s]} (n={len(v)})')
ax[0].set(xlabel='outbound duration (s)', ylabel='trial count', title='Distribution')
ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

ax[1].boxplot([a.values, b.values], tick_labels=[LABELS[s] for s in SESSIONS],
              showfliers=False, widths=.5, medianprops=dict(color='black', lw=2))
rng = np.random.default_rng(0)
for i, (s, v) in enumerate([(SESSIONS[0], a), (SESSIONS[1], b)], start=1):
    ax[1].scatter(rng.normal(i, .06, len(v)), v, alpha=.55, s=22, color=COLOURS[s],
                  zorder=3, edgecolors='white', linewidths=.3)
ax[1].set(ylabel='outbound duration (s)', title='Per-trial'); ax[1].grid(alpha=.3, axis='y')

for s, v in [(SESSIONS[0], a), (SESSIONS[1], b)]:
    xs = np.sort(v.values); ys = np.arange(1, len(xs) + 1) / len(xs)
    ax[2].step(xs, ys, where='post', color=COLOURS[s], lw=2, label=f'{LABELS[s]}  median={v.median():.1f}s')
ax[2].set(xlabel='outbound duration (s)', ylabel='cumulative fraction', title=f'ECDF (Mann-Whitney p={p_mwu:.2g})')
ax[2].legend(fontsize=9); ax[2].grid(alpha=.3)

fig.suptitle(f'{MOUSE}  ·  Test  ·  Day 44  ·  outbound-path duration (start -> target-zone triggered)', y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
#=== 2. Within-session trend - does the outbound duration climb across trials (fatigue / drift)?
# Spearman(outbound duration vs trial #) + first-half vs second-half medians and success rate.
for s in SESSIONS:
    d = sub[s]; half = len(d) // 2
    first, second = d.iloc[:half], d.iloc[half:]
    rho, p = stats.spearmanr(d['trial_index'], d['outbound_dur'])
    succ = lambda x: (x['outcome'] == 'Success').mean()
    print(f'{LABELS[s]} ({len(d)} trials)')
    print(f'  Spearman(outbound dur vs trial #): rho={rho:+.2f}, p={p:.3g}')
    print(f'  1st half: median={first["outbound_dur"].median():6.2f}s  success={succ(first):.0%}')
    print(f'  2nd half: median={second["outbound_dur"].median():6.2f}s  success={succ(second):.0%}\n')

# --- Time course: outbound duration across trials (line per session; marker = outcome).
mk = {'Success': 'o', 'Failure': 'v', 'Miss': 'x'}
fig, ax = plt.subplots(figsize=(11, 4.4))
for s in SESSIONS:
    d = sub[s]
    ax.plot(d['trial_index'], d['outbound_dur'], '-', color=COLOURS[s], alpha=.45, lw=1.1, zorder=1)
    for oc, m in mk.items():
        dd = d[d['outcome'] == oc]
        ax.scatter(dd['trial_index'], dd['outbound_dur'], marker=m, s=34, color=COLOURS[s], zorder=2)
handles = ([Line2D([0], [0], color=COLOURS[s], lw=2, label=LABELS[s]) for s in SESSIONS]
           + [Line2D([0], [0], marker=m, color='0.35', ls='', label=oc) for oc, m in mk.items()])
ax.legend(handles=handles, fontsize=8, ncol=2)
ax.set(xlabel='trial index', ylabel='outbound duration (s)', title='Outbound duration across the session')
ax.grid(alpha=.25); fig.tight_layout(); plt.show()

In [ ]:
#=== 3. Success rate as a function of outbound duration - binned
# Bin trials by outbound duration; compare the success rate per bin. Matched test = chi2 of
# success vs session restricted to bins with >=4 trials in BOTH sessions.
edges  = np.array([0, 15, 20, 30, 50, np.inf])
binlbl = ['<15', '15-20', '20-30', '30-50', '>50']
t['dur_bin'] = pd.cut(t['outbound_dur'], bins=edges, labels=binlbl, right=False)

g = (t.groupby(['dur_bin', 'session'], observed=True)
     .agg(n=('success', 'size'), succ=('success', 'mean')).reset_index())
piv_n = g.pivot(index='dur_bin', columns='session', values='n')
piv_r = g.pivot(index='dur_bin', columns='session', values='succ')
piv_n.columns = [LABELS[c] for c in piv_n.columns]; piv_r.columns = [LABELS[c] for c in piv_r.columns]
print('n trials per bin:'); print(piv_n.fillna(0).astype(int).to_string(), '\n')
print('success rate per bin (%):'); print((piv_r * 100).round(0).astype('Int64').to_string())

both = piv_n.dropna()[(piv_n.dropna() >= 4).all(axis=1)]
if len(both):
    sd = t[t['dur_bin'].astype(str).isin([str(i) for i in both.index])]
    ct = pd.crosstab(sd['session'], sd['success'])
    if ct.shape == (2, 2) and (ct.sum(axis=1) > 0).all():
        chi2, p_match, _, _ = stats.chi2_contingency(ct)
        print(f'\nMatched on outbound duration (bins {list(both.index)}): chi2 p(success vs session) = {p_match:.3g}')
    else:
        print('\nMatched bins found, but success is (near-)constant in one session -> chi2 not meaningful.')
else:
    print('\nNo duration bin has >=4 trials in BOTH sessions -> no matched comparison (the sessions barely overlap).')

# --- Plot 2: success rate per duration bin
fig, ax = plt.subplots(figsize=(9, 4.6))
x, w = np.arange(len(binlbl)), 0.38
for i, s in enumerate(SESSIONS):
    d = t[t['session'] == s]
    rates = [d.loc[d['dur_bin'] == bl, 'success'].mean() if (d['dur_bin'] == bl).any() else np.nan for bl in binlbl]
    ns    = [int((d['dur_bin'] == bl).sum()) for bl in binlbl]
    ax.bar(x + (i - .5) * w, rates, w, color=COLOURS[s], alpha=.85, label=LABELS[s])
    for xi, r, nn in zip(x + (i - .5) * w, rates, ns):
        if nn:
            ax.text(xi, (r if np.isfinite(r) else 0) + .02, f'n={nn}', ha='center', va='bottom', fontsize=8)
ax.set(xticks=x, xlabel='outbound duration (s)', ylabel='success rate', ylim=(0, 1.08),
       title='Success rate as a function of outbound duration')
ax.set_xticklabels(binlbl); ax.legend(); ax.grid(alpha=.3, axis='y')
fig.tight_layout(); plt.show()

---
## Definition B (NEW) — outbound = start → target-zone **available**

**This is the new definition.** Sections **B1–B3** mirror A1–A3 but on **`outbound_dur_avail`**
(`tz_available_time − start_time`): trial start → the moment the target zone is *announced/available*,
i.e. the **pre-availability phase**. This is a *different leg* from Definition A — it excludes the
approach `available → triggered` — so the two are not directly comparable magnitudes. Unlike A it is
defined even on a true Miss.

*(For these two sessions: available-based outbound is S1 median ≈ 11.8 s, S2 median ≈ 55.6 s — shorter
and tighter than the triggered-based version, which is why B3 uses its own bin edges.)*

In [ ]:
#=== [NEW DEF · B1] Outbound(start->available) distribution — summary stats + Mann-Whitney
DURCOL = 'outbound_dur_avail'   # <-- start -> target-zone AVAILABLE  (the NEW definition)
def describe(v, s):
    v = v.dropna()
    return {'session': LABELS[s], 'n': len(v), 'mean': v.mean(), 'median': v.median(),
            'std': v.std(), 'q25': v.quantile(.25), 'q75': v.quantile(.75), 'min': v.min(), 'max': v.max()}

a, b = sub[SESSIONS[0]][DURCOL].dropna(), sub[SESSIONS[1]][DURCOL].dropna()
summary = pd.DataFrame([describe(sub[s][DURCOL], s) for s in SESSIONS]).set_index('session')
u, p_mwu = stats.mannwhitneyu(a, b, alternative='two-sided')

pd.set_option('display.float_format', lambda x: f'{x:.2f}')
print('OUTBOUND DURATION [NEW DEF] (start -> target-zone AVAILABLE), seconds\n')
print(summary.to_string())
print(f'\nMann-Whitney U: U={u:.0f}, p={p_mwu:.3g}  |  '
      f'median {a.median():.2f}s vs {b.median():.2f}s  ({b.median() / a.median():.2f}x)')

# --- Plot: histogram, box + strip, ECDF
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
bins = np.linspace(0, np.nanpercentile(pd.concat([a, b]), 99) * 1.05, 25)
for s, v in [(SESSIONS[0], a), (SESSIONS[1], b)]:
    ax[0].hist(v, bins=bins, alpha=.55, color=COLOURS[s], edgecolor='white', label=f'{LABELS[s]} (n={len(v)})')
ax[0].set(xlabel='outbound duration (s)', ylabel='trial count', title='Distribution')
ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

ax[1].boxplot([a.values, b.values], tick_labels=[LABELS[s] for s in SESSIONS],
              showfliers=False, widths=.5, medianprops=dict(color='black', lw=2))
rng = np.random.default_rng(0)
for i, (s, v) in enumerate([(SESSIONS[0], a), (SESSIONS[1], b)], start=1):
    ax[1].scatter(rng.normal(i, .06, len(v)), v, alpha=.55, s=22, color=COLOURS[s],
                  zorder=3, edgecolors='white', linewidths=.3)
ax[1].set(ylabel='outbound duration (s)', title='Per-trial'); ax[1].grid(alpha=.3, axis='y')

for s, v in [(SESSIONS[0], a), (SESSIONS[1], b)]:
    xs = np.sort(v.values); ys = np.arange(1, len(xs) + 1) / len(xs)
    ax[2].step(xs, ys, where='post', color=COLOURS[s], lw=2, label=f'{LABELS[s]}  median={v.median():.1f}s')
ax[2].set(xlabel='outbound duration (s)', ylabel='cumulative fraction', title=f'ECDF (Mann-Whitney p={p_mwu:.2g})')
ax[2].legend(fontsize=9); ax[2].grid(alpha=.3)

fig.suptitle(f'{MOUSE}  ·  Test  ·  Day 44  ·  [NEW DEF] outbound duration (start -> target-zone available)', y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
#=== [NEW DEF · B2] Within-session trend — does outbound(start->available) climb across trials?
DURCOL = 'outbound_dur_avail'   # start -> target-zone AVAILABLE  (the NEW definition)
for s in SESSIONS:
    d = sub[s]; half = len(d) // 2
    first, second = d.iloc[:half], d.iloc[half:]
    rho, p = stats.spearmanr(d['trial_index'], d[DURCOL])
    succ = lambda x: (x['outcome'] == 'Success').mean()
    print(f'{LABELS[s]} ({len(d)} trials)')
    print(f'  Spearman(outbound[avail] vs trial #): rho={rho:+.2f}, p={p:.3g}')
    print(f'  1st half: median={first[DURCOL].median():6.2f}s  success={succ(first):.0%}')
    print(f'  2nd half: median={second[DURCOL].median():6.2f}s  success={succ(second):.0%}\n')

# --- Time course: outbound(start->available) across trials (line per session; marker = outcome).
mk = {'Success': 'o', 'Failure': 'v', 'Miss': 'x'}
fig, ax = plt.subplots(figsize=(11, 4.4))
for s in SESSIONS:
    d = sub[s]
    ax.plot(d['trial_index'], d[DURCOL], '-', color=COLOURS[s], alpha=.45, lw=1.1, zorder=1)
    for oc, m in mk.items():
        dd = d[d['outcome'] == oc]
        ax.scatter(dd['trial_index'], dd[DURCOL], marker=m, s=34, color=COLOURS[s], zorder=2)
handles = ([Line2D([0], [0], color=COLOURS[s], lw=2, label=LABELS[s]) for s in SESSIONS]
           + [Line2D([0], [0], marker=m, color='0.35', ls='', label=oc) for oc, m in mk.items()])
ax.legend(handles=handles, fontsize=8, ncol=2)
ax.set(xlabel='trial index', ylabel='outbound duration (s)',
       title='[NEW DEF] Outbound (start -> available) across the session')
ax.grid(alpha=.25); fig.tight_layout(); plt.show()

In [ ]:
#=== [NEW DEF · B3] Success rate as a function of outbound(start->available) — binned
DURCOL = 'outbound_dur_avail'   # start -> target-zone AVAILABLE  (the NEW definition)
# Available-based durations run shorter/tighter than triggered, so these edges differ from Def A's
# (chosen from the observed range here: S1 ~7-21s, S2 ~8-97s). Retune if you change sessions.
edges  = np.array([0, 12, 20, 40, 70, np.inf])
binlbl = ['<12', '12-20', '20-40', '40-70', '>70']
t['dur_bin_avail'] = pd.cut(t[DURCOL], bins=edges, labels=binlbl, right=False)

g = (t.groupby(['dur_bin_avail', 'session'], observed=True)
     .agg(n=('success', 'size'), succ=('success', 'mean')).reset_index())
piv_n = g.pivot(index='dur_bin_avail', columns='session', values='n')
piv_r = g.pivot(index='dur_bin_avail', columns='session', values='succ')
piv_n.columns = [LABELS[c] for c in piv_n.columns]; piv_r.columns = [LABELS[c] for c in piv_r.columns]
print('n trials per bin:'); print(piv_n.fillna(0).astype(int).to_string(), '\n')
print('success rate per bin (%):'); print((piv_r * 100).round(0).astype('Int64').to_string())

both = piv_n.dropna()[(piv_n.dropna() >= 4).all(axis=1)]
if len(both):
    sd = t[t['dur_bin_avail'].astype(str).isin([str(i) for i in both.index])]
    ct = pd.crosstab(sd['session'], sd['success'])
    if ct.shape == (2, 2) and (ct.sum(axis=1) > 0).all():
        chi2, p_match, _, _ = stats.chi2_contingency(ct)
        print(f'\nMatched on outbound[avail] (bins {list(both.index)}): chi2 p(success vs session) = {p_match:.3g}')
    else:
        print('\nMatched bins found, but success is (near-)constant in one session -> chi2 not meaningful.')
else:
    print('\nNo duration bin has >=4 trials in BOTH sessions -> no matched comparison (the sessions barely overlap).')

# --- Plot: success rate per duration bin
fig, ax = plt.subplots(figsize=(9, 4.6))
x, w = np.arange(len(binlbl)), 0.38
for i, s in enumerate(SESSIONS):
    d = t[t['session'] == s]
    rates = [d.loc[d['dur_bin_avail'] == bl, 'success'].mean() if (d['dur_bin_avail'] == bl).any() else np.nan for bl in binlbl]
    ns    = [int((d['dur_bin_avail'] == bl).sum()) for bl in binlbl]
    ax.bar(x + (i - .5) * w, rates, w, color=COLOURS[s], alpha=.85, label=LABELS[s])
    for xi, r, nn in zip(x + (i - .5) * w, rates, ns):
        if nn:
            ax.text(xi, (r if np.isfinite(r) else 0) + .02, f'n={nn}', ha='center', va='bottom', fontsize=8)
ax.set(xticks=x, xlabel='outbound duration (s)', ylabel='success rate', ylim=(0, 1.08),
       title='[NEW DEF] Success rate vs outbound (start -> available)')
ax.set_xticklabels(binlbl); ax.legend(); ax.grid(alpha=.3, axis='y')
fig.tight_layout(); plt.show()